# 0.7 Adaptation Protection Prep

This prep notebook builds a raster-explicit protection adaptation scenario
from basin losses that already include both `damages` and `adapted_damages`,
then writes a scenario basin risk CSV for the generic simulation notebooks.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import pandas as pd

from sovereign.flood import build_protection_scenario_risk_data


In [ ]:
# USER CONFIG
model = "wri"
scenario_name = "urban_protection_aep001"
adapted_protection_aep = 0.01


In [ ]:
# Paths and baseline inputs
root = Path.cwd().parent
risk_basin_path = root / "outputs" / "flood" / "risk" / "basins" / f"risk_basins_m-{model}.csv"

scenario_root = root / "outputs" / "flood" / "adaptation" / "protection" / scenario_name
scenario_basin_dir = scenario_root / "basins"
scenario_basin_dir.mkdir(parents=True, exist_ok=True)

risk_data = pd.read_csv(risk_basin_path)
risk_data = risk_data.iloc[:, 1:] if str(risk_data.columns[0]).startswith("Unnamed") else risk_data
risk_data["AEP"] = 1 / risk_data["RP"]
risk_data["Pr_L_AEP"] = np.where(risk_data["Pr_L"] == 0, 0, 1 / risk_data["Pr_L"])
risk_data.reset_index(drop=True, inplace=True)

if "adapted_damages" not in risk_data.columns:
    raise ValueError(
        "Protection prep expects the baseline basin risk CSV to include an adapted_damages column."
    )

# The adapted dataframe reuses the same keys but exposes raster-derived adapted losses as `damages`
adapted_risk_data = risk_data.copy()
adapted_risk_data["damages"] = adapted_risk_data["adapted_damages"]

risk_data.head()


In [ ]:
# Build and save scenario basin dataframe
scenario_risk_df = build_protection_scenario_risk_data(
    baseline_risk_df=risk_data,
    adapted_risk_df=adapted_risk_data,
    adapted_protection_aep=adapted_protection_aep,
)

scenario_basin_path = scenario_basin_dir / f"risk_basins_m-{model}.csv"
scenario_risk_df.to_csv(scenario_basin_path, index=False)

print("Scenario basin CSV written to:")
print(scenario_basin_path)
scenario_risk_df.head()


## Notes

This notebook assumes the baseline basin risk CSV already contains a
raster-derived `adapted_damages` column from the protection workflow.

To customize this scenario, change:

- `scenario_name`
- `adapted_protection_aep`

The output CSV includes event-window fields so it can be read directly by
`3_national_flood_simulation_scenario.ipynb` and
`4_macro_simulation_scenario.ipynb`.
